[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-learning-theory.ipynb)

# Why Learning Works — Generalisation, VC Dimension and Information Theory

*AIBits Academy · Machine Learning End To End · Advanced · Optional*

A model trained on a few hundred rows makes claims about millions of customers it has never seen. This page shows, with small experiments, when that is justified and how much data it takes.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **⚙ Optional page**
>
> This page is less directly useful to a job-oriented reader than the rest of the course. Treat it as an optional advanced page: skipping it does not affect any project, and you can return to it when you want to understand *why* cross-validation, regularisation, model complexity and log-loss behave as they do.

## The Puzzle

Every model in this course minimises error on the training data. But the training error is not what we care about: we care about the **true error**, the error on the whole population the data came from. The difference, the *generalisation gap*, is the price of having fitted a finite sample. Learning theory asks: when is that gap small, and how does it depend on the amount of data and the flexibility of the model?

## A Finite Menu: the Union Bound

Start with the simplest hypothesis class we can imagine: 101 possible cut-off rules “approve if the score is at least *t*” for *t* = 0, 0.01, …, 1. The real rule is a cut-off at 0.6, but 10% of the labels are flipped by noise, so even the best rule has a true error of 0.10. Empirical risk minimisation (ERM) picks the cut-off with the lowest training error. Hoeffding’s inequality says that for any *one* fixed rule the training error is within ε of its true error with high probability; a union bound over all |H| rules gives, with probability at least 1 − δ, that *every* rule in H satisfies:

$$\left|\,R(h)-\hat R(h)\,\right| \le \sqrt{\dfrac{\ln|H| + \ln(2/\delta)}{2n}}$$

Here *R* is the true error and *R̂* the training error. The size of the menu enters only through ln|H|: doubling the number of hypotheses adds a constant. We ran ERM 3,000 times at each sample size and compared the 95th-percentile gap with this bound:

In [ ]:
import numpy as np

rng = np.random.RandomState(0)
thresholds = np.linspace(0, 1, 101)                                     # the hypothesis class H: 101 possible cut-offs, "approve if score >= t"
K = len(thresholds); delta = 0.05
true_cut, noise = 0.6, 0.10                                             # the real rule is a cut-off at 0.6, but 10% of labels are flipped by noise

def true_error(t):
    """Exact error of cut-off t on the whole population: it disagrees with the real rule on |t - 0.6| of the scores, and noise hits the rest."""
    return noise + (1 - 2 * noise) * abs(t - true_cut)

print("empirical risk minimisation over 101 thresholds; 3,000 repetitions per sample size")
print("     n    train error   true error   gap (95th pct)   Hoeffding+union bound   bound holds?")
for n in (10, 30, 100, 300, 1000, 3000):
    gaps, tr_err, tru_err = [], [], []
    for _ in range(3000):
        x = rng.rand(n); y = (x >= true_cut) ^ (rng.rand(n) < noise)   # labels: the real rule, with 10% flipped
        train = np.array([np.mean((x >= t) != y) for t in thresholds])   # error of every hypothesis on the sample
        best = train.argmin()                                            # ERM: pick the hypothesis with the lowest training error
        tr_err.append(train[best]); tru_err.append(true_error(thresholds[best])); gaps.append(true_error(thresholds[best]) - train[best])
    bound = np.sqrt((np.log(K) + np.log(2 / delta)) / (2 * n))           # with prob >= 1-delta, EVERY h in H has |true - train| <= bound
    g95 = np.percentile(gaps, 95)
    print(f"{n:6d}     {np.mean(tr_err):.3f}        {np.mean(tru_err):.3f}        {g95:.3f}            {bound:.3f}               {'yes' if g95 <= bound else 'NO'}")
print(f"\n(the best possible true error is the label noise, {noise:.2f}; the bound grows only with ln|H| = {np.log(K):.2f}, not with |H|)")

| Sample size n | Train error | True error | Gap (95th pct) | Bound | Holds? |
|---|---|---|---|---|---|
| 10 | 0.079 | 0.197 | 0.316 | 0.644 | yes |
| 30 | 0.093 | 0.134 | 0.139 | 0.372 | yes |
| 100 | 0.098 | 0.108 | 0.060 | 0.204 | yes |
| 300 | 0.100 | 0.102 | 0.030 | 0.118 | yes |
| 1,000 | 0.100 | 0.100 | 0.015 | 0.064 | yes |
| 3,000 | 0.100 | 0.100 | 0.009 | 0.037 | yes |

Three lessons. With 10 points the chosen rule looks excellent (training error 0.079, below the unavoidable noise level of 0.10) but has a true error of 0.197: **the training error of a selected model is optimistic** — the more hypotheses you compare, the more optimistic it is. The gap falls like 1/√*n* (0.316 at 10, 0.060 at 100, 0.009 at 3,000). And the bound is always valid but **loose**: at n = 100 it promises 0.204 when the observed gap is 0.060. Theory tells you how things scale, not the exact size of your error.

## Beyond a Finite Menu: VC Dimension

Real models have infinitely many hypotheses (every possible line), so |H| is useless. Vapnik and Chervonenkis measured richness differently: a set of points is **shattered** by a hypothesis class if the class can realise *every* possible labelling of them. The **VC dimension** *d* is the largest number of points the class can shatter. We can check this by brute force — for lines in the plane, testing linear separability of each labelling with a tiny linear programme:

In [ ]:
import itertools
import numpy as np
from scipy.optimize import linprog

def separable(points, labels):
    """Is there a line w.x + b that puts +1 points strictly on one side and -1 points strictly on the other? (a tiny linear programme)"""
    P = np.asarray(points, float); s = np.where(labels, 1.0, -1.0)
    A = -(s[:, None] * np.column_stack([P, np.ones(len(P))]))           # need  s_i * (w.x_i + b) >= 1
    res = linprog(np.zeros(P.shape[1] + 1), A_ub=A, b_ub=-np.ones(len(P)), bounds=[(None, None)] * (P.shape[1] + 1))
    return res.status == 0

def shatter_count_lines(points):
    return sum(separable(points, np.array(lab)) for lab in itertools.product([0, 1], repeat=len(points)))

def interval_realisable(xs, labels):
    """1-D intervals [a,b]: the positives must be one consecutive block in sorted order (or absent)."""
    order = np.argsort(xs); l = np.asarray(labels)[order]; pos = np.where(l == 1)[0]
    return len(pos) == 0 or np.all(np.diff(pos) == 1)

print("Lines in the plane (perceptron-style classifiers):")
for name, pts in [("2 points", [(0, 0), (1, 1)]), ("3 points, general position", [(0, 0), (1, 0), (0, 1)]), ("3 points on one line", [(0, 0), (1, 1), (2, 2)]),
                  ("4 points: corners of a square", [(0, 0), (1, 0), (0, 1), (1, 1)]), ("4 points: a triangle with a centre point", [(0, 0), (4, 0), (0, 4), (1, 1)])]:
    print(f"  {name:42s} labelings realised: {shatter_count_lines(pts):2d} of {2 ** len(pts)}")

print("\nIntervals on a line (one block of positives):")
for k in (1, 2, 3, 4):
    xs = np.arange(k, dtype=float); got = sum(interval_realisable(xs, lab) for lab in itertools.product([0, 1], repeat=k))
    print(f"  {k} points: {got:2d} of {2 ** k} labelings realised")
print("\nVC dimension = the largest number of points that CAN be shattered (all labelings realised): lines in the plane -> 3, intervals -> 2")

Lines in the plane shatter three points in general position (all 8 labellings) but not four: the two “XOR” labellings of a square’s corners cannot be separated, so only 14 of 16 are realised. Three points on a straight line fail too (6 of 8). So the VC dimension of lines in the plane is **3**. Intervals on a line shatter two points but not three (7 of 8: the pattern “in, out, in” is impossible), so their VC dimension is **2**. In general, a linear classifier in *p* dimensions has VC dimension *p* + 1 — roughly, the number of parameters. Vapnik’s bound then says that with probability at least 1 − δ:

$$R(h) \le \hat R(h) + \sqrt{\dfrac{d\left(\ln\frac{2n}{d}+1\right)+\ln\frac{4}{\delta}}{n}}$$

In [ ]:
import numpy as np

def vc_bound(d, n, delta=0.05):
    """Vapnik's bound: with probability >= 1 - delta, true error <= training error + this amount."""
    return np.sqrt((d * (np.log(2 * n / d) + 1) + np.log(4 / delta)) / n)

print("  VC dimension d   n=100   n=1,000   n=10,000   n=100,000")
for d in (3, 11, 51, 501):
    print(f"  {d:12d}    " + "   ".join(f"{vc_bound(d, n):6.3f}" if n > d else "   n/a " for n in (100, 1_000, 10_000, 100_000)))
print("\nrule of thumb from the bound: the gap shrinks roughly like sqrt(d / n); to halve it you need about four times the data")
print("n needed for a gap below 0.05 with 95% confidence:", {d: int(next(n for n in range(d + 1, 10_000_000, max(1, d // 3)) if vc_bound(d, n) < 0.05)) for d in (3, 11, 51)})

For a model with VC dimension 3 the guaranteed gap is 0.164 at n = 1,000 and 0.058 at 10,000; with VC dimension 51 it is 0.492 at n = 1,000, and for d = 501 the bound says nothing (1.095) until you have tens of thousands of rows. Guaranteeing a gap under 0.05 needs **13,916** rows for d = 3, **45,882** for d = 11 and **205,667** for d = 51. These worst-case guarantees are very conservative (real learners do far better on friendly data), but the shape is correct: the gap grows with model richness and shrinks with √*n*.

## The Bias–Complexity Trade-Off in Numbers

Fit polynomials of increasing degree to noisy samples of a sine curve. The best any model can do on new data is the noise variance, 0.09. Averages over 200 training sets:

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

rng = np.random.RandomState(3)
f = lambda x: np.sin(2 * np.pi * x)                                     # the true curve
noise_sd = 0.3
xte = rng.rand(5000); yte = f(xte) + rng.normal(0, noise_sd, 5000)       # a big fresh test set stands in for the population

print("Mean squared error (average of 200 training sets); the best possible test error is the noise variance 0.09")
for n in (20, 200):
    print(f"\ntraining set of {n} points")
    print("  degree   parameters   train MSE   test MSE   gap (test - train)")
    for d in (1, 3, 5, 9, 15):
        tr, te = [], []
        for _ in range(200):
            x = rng.rand(n); y = f(x) + rng.normal(0, noise_sd, n)
            m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x[:, None], y)
            tr.append(np.mean((m.predict(x[:, None]) - y) ** 2)); te.append(np.mean((m.predict(xte[:, None]) - yte) ** 2))
        print(f"  {d:5d}    {d + 1:5d}        {np.mean(tr):.3f}      {np.mean(te):.3f}      {np.mean(te) - np.mean(tr):+.3f}")

| Degree (parameters) | Train MSE, n=20 | Test MSE, n=20 | Train MSE, n=200 | Test MSE, n=200 |
|---|---|---|---|---|
| 1 (2) | 0.251 | 0.310 | 0.285 | 0.281 |
| 3 (4) | 0.075 | 0.145 | 0.091 | 0.096 |
| 5 (6) | 0.063 | 0.594 | 0.087 | 0.093 |
| 9 (10) | 0.050 | **202.417** | 0.087 | 0.094 |
| 15 (16) | 0.046 | **1334.141** | 0.085 | 0.096 |

With 20 points, training error keeps falling as the degree rises (0.251 down to 0.046) while test error explodes (0.145 at degree 3, 202 at degree 9, more than 1,300 at degree 15; the averages are dominated by a few training sets whose high-degree fit swings wildly between points). With 200 points the same models are harmless: test error stays near 0.09–0.10 for every degree from 3 to 15, and the gap between training and test error is at most 0.011. **More data buys the right to use a richer model**, which is the practical content of the VC bound. A rough working rule: keep the number of free parameters well below the number of training rows (a tenth or less unless you regularise), and use validation data to check.

## Information Theory: the Language of Loss Functions

Three quantities link probability to the loss functions you have been minimising. The **entropy** of a distribution is the average number of bits needed to describe an outcome when you use the best possible code for it; the **cross-entropy** is the average number of bits when you use a code built for a *different* distribution *q*; and their difference is the **KL divergence**, the bits wasted by believing *q* when the truth is *p*:

$$H(p) = -\sum_i p_i \log_2 p_i \qquad H(p,q) = -\sum_i p_i \log_2 q_i \qquad \operatorname{KL}(p\,\Vert\,q) = H(p,q)-H(p) \ge 0$$

In [ ]:
import numpy as np
from scipy.stats import entropy
from sklearn.metrics import log_loss

def H(p):  return -np.sum([q * np.log2(q) for q in p if q > 0])                 # entropy in bits
def CE(p, q): return -np.sum([a * np.log2(b) for a, b in zip(p, q) if a > 0])     # cross-entropy in bits
def KL(p, q): return CE(p, q) - H(p)                                             # KL divergence = extra bits paid for using the wrong code

print("Entropy of a loan outcome (bits per customer):")
for p in (0.5, 0.12, 0.02):
    print(f"  P(default) = {p:.2f}: H = {H([p, 1 - p]):.3f} bits   (scipy: {entropy([p, 1 - p], base=2):.3f})")

true = [0.12, 0.88]
for name, model in [("a model that knows the truth", [0.12, 0.88]), ("a model that says 20% for everyone", [0.20, 0.80]), ("an over-confident model (2%)", [0.02, 0.98])]:
    print(f"\n{name}: cross-entropy {CE(true, model):.3f} bits = entropy {H(true):.3f} + KL {KL(true, model):.3f}")

# cross-entropy IS log-loss: check against scikit-learn on a small labelled sample
rng = np.random.RandomState(0); y = (rng.rand(20000) < 0.12).astype(int); p = np.full(20000, 0.20)
print(f"\nlog-loss of the constant 20% model on 20,000 simulated customers: sklearn {log_loss(y, p):.4f} nats = {log_loss(y, p) / np.log(2):.3f} bits (theory {CE(true, [0.2, 0.8]):.3f})")

# information gain of a split, as a decision tree computes it
parent = [0.30, 0.70]; left, right = [0.60, 0.40], [0.10, 0.90]; wl = 0.4
gain = H(parent) - (wl * H(left) + (1 - wl) * H(right))
print(f"\nSplit: parent {parent} -> left ({wl:.0%} of rows) {left}, right {right}: information gain = {H(parent):.3f} - {wl * H(left) + (1 - wl) * H(right):.3f} = {gain:.3f} bits")

A loan outcome with a 12% default rate carries only 0.529 bits of uncertainty per customer (a coin flip would carry 1.000). A model that knows the truth pays exactly that; a model that says “20% for everyone” pays 0.562, wasting KL = 0.033 bits; an over-confident model that says 2% pays 0.703, wasting 0.174. The **log-loss** that logistic regression and neural classifiers minimise is precisely the cross-entropy between the true labels and the predicted probabilities, measured in nats (natural logarithm) instead of bits: scikit-learn’s value of 0.3991 nats equals 0.576 bits, close to the theoretical 0.562 (the difference is sampling variation in the simulated default rate). Minimising log-loss is therefore the same as minimising KL divergence from the truth — which is why over-confident wrong predictions are punished so heavily. Finally, the **information gain** used by decision trees is just the drop in entropy from a split: here 0.881 bits for the parent node minus 0.670 bits for the weighted children, a gain of **0.212** bits.

> **⚠ What theory does and does not tell you**
>
> The bounds are worst-case and loose: the finite-class bound promised 0.204 where the observed gap was 0.060, and VC bounds are far larger than what practical learners achieve. They say nothing about a specific dataset, and modern over-parameterised models (huge networks) generalise better than VC dimension suggests, which is an active research topic. Use the theory for its scaling laws — the gap shrinks like √(*d*/*n*), the training error of a selected model is optimistic — and use held-out data for the actual numbers.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The union-bound guarantee

Write `hoeffding_bound(K, n, delta)` returning the gap that holds for *every* one of `K` hypotheses with probability at least `1 - delta`: `sqrt((ln K + ln(2/delta)) / (2 n))`. Use it for K = 101, n = 100, delta = 0.05.

In [ ]:
import numpy as np
def hoeffding_bound(K, n, delta):
    return None   # TODO
bound = hoeffding_bound(101, 100, 0.05)


In [ ]:
try:
    check("bound is about 0.204", round(bound, 3) == 0.204)
    check("400 samples halve it", round(hoeffding_bound(101, 400, 0.05), 3) == 0.102)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def hoeffding_bound(K, n, delta):
    return float(np.sqrt((np.log(K) + np.log(2 / delta)) / (2 * n)))
bound = hoeffding_bound(101, 100, 0.05)
```

</details>


### Exercise 2 · Medium · Entropy and the price of a wrong belief

Implement `entropy(p)` and `kl(p, q)` in **bits** (log base 2), then compute the entropy of a 12% default rate and the KL divergence to a model that says 20%.

In [ ]:
import numpy as np
def entropy(p): return None   # TODO
def kl(p, q): return None      # TODO
h = entropy([0.12, 0.88]); d = kl([0.12, 0.88], [0.20, 0.80])


In [ ]:
try:
    check("entropy is 0.529 bits", round(h, 3) == 0.529)
    check("KL is 0.033 bits", round(d, 3) == 0.033)
    check("KL of a distribution with itself is 0", abs(kl([0.3, 0.7], [0.3, 0.7])) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def entropy(p):
    p = np.asarray(p, float); return float(-(p * np.log2(p)).sum())
def kl(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float); return float((p * np.log2(p / q)).sum())
h = entropy([0.12, 0.88]); d = kl([0.12, 0.88], [0.20, 0.80])
```

</details>


### Exercise 3 · Stretch · Can lines shatter a square?

`separable(points, labels)` (given) says whether a straight line can put the `True` points on one side and the `False` points on the other. Count how many of the 16 labellings of the four corners of a square it can realise, in `n_realised`.

In [ ]:
import itertools, numpy as np
from scipy.optimize import linprog
def separable(points, labels):
    P = np.asarray(points, float); s = np.where(labels, 1.0, -1.0)
    A = -(s[:, None] * np.column_stack([P, np.ones(len(P))]))
    return linprog(np.zeros(3), A_ub=A, b_ub=-np.ones(len(P)), bounds=[(None, None)] * 3).status == 0
square = [(0, 0), (1, 0), (0, 1), (1, 1)]
n_realised = None   # TODO: loop over itertools.product([False, True], repeat=4)


In [ ]:
try:
    check("14 of 16 labellings", n_realised == 14)
    check("so lines cannot shatter four points", n_realised < 16)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
n_realised = sum(separable(square, np.array(lab)) for lab in itertools.product([False, True], repeat=4))
```

</details>


---
*Back to the course: **Machine Learning End To End → Why Learning Works — Generalisation, VC Dimension and Information Theory**.*